# Kaggle/Dacon Pipeline Runner

This notebook serves as the main entry point for running the pipeline on Kaggle or other cloud environments. It automatically detects the environment and adjusts paths via the `config.yaml`.

In [ ]:
# Removed autoreload to prevent 'imp' module error in Python 3.12

In [ ]:
!pip install -q optuna pytorch-lightning omegaconf catboost

In [ ]:
!ls -laR /kaggle/input || true
!pwd

In [ ]:
import os
import sys
import warnings
from pathlib import Path
import torch

warnings.filterwarnings('ignore')

# Fix for Kaggle P100 GPU PyTorch LSTM bug (CUDA error: no kernel image is available)
torch.backends.cudnn.enabled = False

# Detect if running on Kaggle
is_kaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') != ''

if is_kaggle:
    # Auto-discover the code dataset path
    input_dir = Path('/kaggle/input')
    code_dataset_path = None
    for p in input_dir.rglob('conf/config.yaml'):
        code_dataset_path = p.parents[1]
        break
        
    if code_dataset_path is None:
        raise FileNotFoundError("Could not find conf/config.yaml inside /kaggle/input")
        
    CODE_DATASET_PATH = code_dataset_path
    if str(CODE_DATASET_PATH) not in sys.path:
        sys.path.append(str(CODE_DATASET_PATH))
    # Kaggle default working directory is /kaggle/working
    PROJECT_ROOT = Path('/kaggle/working')
    print(f"Running on Kaggle. Added {CODE_DATASET_PATH} to sys.path")
else:
    # If running locally, make sure the project root is in sys.path
    PROJECT_ROOT = Path.cwd().resolve().parents[1]
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.append(str(PROJECT_ROOT))
    os.chdir(PROJECT_ROOT)
    print(f"Running locally. Changed working directory to {PROJECT_ROOT}")

In [ ]:
import yaml

# Load Config
conf_path = Path('conf/config.yaml')
if is_kaggle:
    conf_path = CODE_DATASET_PATH / 'conf/config.yaml'

with open(conf_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# Override environment if running on Kaggle
if is_kaggle:
    config['env'] = 'kaggle'
    
    # Auto-discover data_dir
    data_dir = None
    for p in Path('/kaggle/input').rglob('ch2026_metrics_train.csv'):
        data_dir = p.parent
        break
        
    if data_dir is not None:
        config['paths']['kaggle']['data_dir'] = str(data_dir)
        print(f"Auto-discovered data_dir: {data_dir}")

env = config['env']
paths = config['paths'][env]

print(f"Environment: {env}")
print(f"Data Dir: {paths['data_dir']}")
print(f"Model Dir: {paths['model_dir']}")

In [ ]:
# Setup output directories
for dir_name in ['processed_dir', 'model_dir', 'output_dir']:
    if dir_name in paths:
        Path(paths[dir_name]).mkdir(parents=True, exist_ok=True)

## Execute Pipeline

Now we can import our modules from `src` and run the pipeline steps.

In [ ]:
from omegaconf import OmegaConf
from src.models.trainer import Trainer
from src.models.dl_trainer import DLTrainer
import sys
from pathlib import Path

# Since config is a dict, we can convert it to DictConfig
cfg = OmegaConf.create(config)

is_dl = cfg.model.get("name", "").startswith("dl_")
is_tuning = cfg.training.get("tune_hyperparameters", False)

if is_dl and is_tuning:
    print("Running Massive DL Hyperparameter Tuning with Optuna...")
    # Add scripts to path so we can import the tuning script
    scripts_dir = str(Path(CODE_DATASET_PATH) / 'scripts' if is_kaggle else Path(PROJECT_ROOT) / 'scripts')
    if scripts_dir not in sys.path:
        sys.path.append(scripts_dir)
    import tune_dl_optuna
    tune_dl_optuna.main()
elif is_dl:
    trainer = DLTrainer(cfg)
    trainer.run()
else:
    trainer = Trainer(cfg)
    trainer.run()

print("Pipeline executed successfully!")